# Bell 态：让两个量子比特关联起来

从 `|00⟩` 开始，先对 q0 使用 H 门，再使用以 q0 为控制、q1 为目标的 CNOT 门：

$$|00\rangle \longrightarrow \frac{|00\rangle+|10\rangle}{\sqrt{2}}
\longrightarrow |\Phi^+\rangle=\frac{|00\rangle+|11\rangle}{\sqrt{2}}.$$

这个 Notebook 只使用 Python 标准库和 qutai。量子态的振幅使用 Python 内置复数，暂不涉及量子电路求导或训练。


In [1]:
from qutai import quantum as qu, visualization

device = "cuda"
dtype = "float32"


## 1. 先约定怎么看量子态

两个量子比特的顺序是 **q0、q1**，q0 是最左边的一位，也是电路图最上面的线。
振幅列表依次对应 `|00⟩、|01⟩、|10⟩、|11⟩`。

`amplitudes` 保存复数振幅；每个振幅的模平方是测到对应结果的概率。所有概率之和为 1。


In [2]:
initial = qu.State(2).to(device, dtype="complex64" if dtype == "float32" else "complex128")
print("初始振幅:", initial.storage.tolist())
print("初始概率:", initial.probabilities())

初始振幅: [(1+0j), 0j, 0j, 0j]
初始概率: [1.0, 0.0, 0.0, 0.0]


## 2. 搭建 Bell 电路

列表里的门按从上到下的顺序执行；图上则从左向右执行。
H 门产生叠加。CNOT 门在控制位为 1 时翻转目标位，所以 `|10⟩` 会变成 `|11⟩`。

这里只描述和画出电路，还没有执行模拟。


In [3]:
circuit = qu.Circuit(2, [
    qu.H(0),
    qu.CNOT(0, 1),
])

visualization.Circuit(circuit, title="Bell 态电路：H → CNOT")


## 3. 先看只经过 H 门的中间状态

q0 有一半概率为 0、一半概率为 1；q1 仍为 0，所以只有 `00` 和 `10` 有非零概率。


In [4]:
superposition = qu.Circuit(2, [qu.H(0)]).run(initial)

visualization.Table(
    [[f"|{index:02b}⟩", f"{amplitude:.6g}", f"{probability:.1%}"]
     for index, (amplitude, probability) in enumerate(zip(superposition.storage.tolist(), superposition.probabilities()))],
    columns=["基态", "振幅", "概率"],
    title="H 门之后",
)

## 4. 执行完整电路

调用 `circuit.run()` 会从全零状态执行，返回一个新的量子态。再次调用会重新开始。
现在 `00` 和 `11` 各有 50% 的理论概率，`01` 和 `10` 的概率为 0。


In [5]:
state = circuit.run(initial)

visualization.Table(
    [[f"|{index:02b}⟩", f"{amplitude:.6g}", f"{probability:.1%}"]
     for index, (amplitude, probability) in enumerate(zip(state.storage.tolist(), state.probabilities()))],
    columns=["基态", "振幅", "概率"],
    title="Bell 态的振幅与概率",
)

In [6]:
visualization.Probabilities(state, title="Bell 态的理论测量概率")

## 5. 如果只看一个量子比特

`wires=0` 表示只看 q0，将其他比特的可能结果合并。单看 q0 或 q1，都各有一半概率为 0 或 1。
完整的联合分布告诉我们：两位的结果总是相同。


In [7]:
print("只看 q0:", state.probabilities(wires=0))
print("只看 q1:", state.probabilities(wires=1))

visualization.Probabilities(state, wires=0, title="只看 q0 的理论概率")

只看 q0: [0.4999999701976776, 0.4999999701976776]
只看 q1: [0.4999999701976776, 0.4999999701976776]


## 6. 重复准备 Bell 态，再测量 1000 次

`sample()` 返回每一次的结果，`counts()` 返回结果的次数统计。它们都模拟对相同状态重复准备、独立测量，**不会改变原状态**。

这里省略 seed，让每次采样自然随机；需要复现时可以显式传入 seed。有限次采样不要求恰好 500 次 `00`、500 次 `11`。


In [8]:
print("前 12 次测量:", state.sample(12))
counts = state.counts(1000)
print("1000 次测量统计:", counts)

visualization.Table(
    [[f"{index:02b}", counts.get(f"{index:02b}", 0), f"{probability:.1%}"]
     for index, probability in enumerate(state.probabilities())],
    columns=["结果", "采样次数", "理论概率"],
    title="采样统计与理论概率",
)

前 12 次测量: ['11', '00', '00', '00', '11', '00', '11', '00', '11', '00', '11', '11']
1000 次测量统计: {'00': 514, '11': 486}


## 7. 真正测量一次，观察状态坍缩

`measure()` 会改变当前量子态。先复制一份 Bell 态，再只测 q0：

- 如果得到 0，整态会变为 `|00⟩`。
- 如果得到 1，整态会变为 `|11⟩`。

然后测 q1，它一定与刚才的 q0 相同。原 Bell 态通过 `copy()` 保留下来，便于对照测量前后的概率。


In [9]:
collapsed = state.copy()
first = collapsed.measure(wires=0)
after_first = collapsed.probabilities()
second = collapsed.measure(wires=1)

print("测量 q0:", first)
print("测量 q0 后的联合概率:", after_first)
print("再测 q1:", second)
print("两次结果相同:", first == second)
print("原 Bell 态仍保留原来的概率:", state.probabilities())

visualization.Probabilities(collapsed, title="测量后的状态：只剩一个可能结果")

测量 q0: 1
测量 q0 后的联合概率: [0.0, 0.0, 0.0, 1.0]
再测 q1: 1
两次结果相同: True
原 Bell 态仍保留原来的概率: [0.4999999701976776, 0.0, 0.0, 0.4999999701976776]


## 8. 用 Bloch 球区分局部纯态和混合态

Bloch 球的坐标是单个比特的 X、Y、Z 期望值。向量长度为 1 时是纯态，落在球内时是混合态。

先看只经过 H 门、尚未纠缠的 q0：它是 `|+⟩`，Bloch 向量为 `(1, 0, 0)`。


In [10]:
visualization.Bloch(superposition, wire=0, title="H 门之后的 q0：局部纯态")


再看 Bell 态中的 q0，它的局部状态是最大混合态，向量为 `(0, 0, 0)`，显示在球心。
这不代表整个 Bell 态失去量子信息：整个系统仍然是纯态，两个比特的关联需要联合观测来描述。

画图不会改变原量子态。修改 `wire=1` 可以查看另一个比特。


In [11]:
visualization.Bloch(state, wire=0, title="Bell 态的 q0：局部最大混合态")


到这里，我们完成了量子态、基础量子门、电路、理论概率、重复采样和测量坍缩的闭环。

接下来打开 `quantum_training.ipynb`，用 RX、RY、RZ 的角度参数，将测量期望值接到现有的自动求导和优化器。
